<a href="https://colab.research.google.com/github/ua408447-ui/Resume-skill-extractor/blob/main/Resume_Skill_Extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
dataturks_resume_entities_for_ner_path = kagglehub.dataset_download('dataturks/resume-entities-for-ner')

print('Data source import complete.')


100%|██████████| 323k/323k [00:00<00:00, 28.9MB/s]

Extracting files...
Data source import complete.


# Named Entity Recognition (NER) for Resume Skill Extraction

This notebook implements a complete pipeline for training a BERT-based NER model to extract skills and other entities from resumes.

## Dataset
- **Format**: DataTurks JSON (one JSON object per line)
- **Structure**: Character-level annotations with `content` and `annotation` fields
- **Model**: bert-base-cased (case-sensitive for distinguishing 'C', 'JAVA', etc.)

## Pipeline Overview
1. Load and preprocess data (character → token-level BIO tags)
2. Tokenize with BERT tokenizer
3. Train using Hugging Face Trainer API
4. Evaluate with entity-level metrics (seqeval)
5. Inference on new resume text

## 1. Install and Import Required Libraries

In [2]:
# Install required packages (uncomment if needed)
!pip install transformers datasets torch scikit-learn seqeval pandas numpy accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=9ae0d1eaff2dfe94b19e27b88dc9d90054989541e8160a94cc391d1a751c2f82
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [3]:
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
import torch
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.9.0+cu126
CUDA available: True


## 2. Data Loading and Preprocessing

### Critical: Character-to-Token Alignment

The DataTurks format uses character-level indices, but BERT tokenization operates on tokens (subwords). We need to:
1. Parse each JSON line to extract resume text and annotations
2. Convert character spans to token spans
3. Apply BIO tagging scheme (B-LABEL, I-LABEL, O)
4. Handle edge cases where entity boundaries don't align with token boundaries

In [4]:
def load_dataturks_json(file_path: str) -> List[Dict]:
    """
    Load DataTurks JSON format where each line is a separate JSON object.

    Args:
        file_path: Path to the .json file

    Returns:
        List of dictionaries containing 'content' and 'annotation'
    """
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"Skipping invalid JSON line: {e}")
    print(f"Loaded {len(data)} documents")
    return data

In [5]:
def char_to_token_alignment(text: str, annotations: List[Dict], tokenizer) -> Tuple[List[str], List[str]]:
    """
    Convert character-level annotations to token-level BIO tags.

    This is the most critical function for NER preprocessing.

    Args:
        text: Raw resume text
        annotations: List of annotation dicts with 'label' and 'points' (char indices)
        tokenizer: BERT tokenizer

    Returns:
        tokens: List of tokens
        tags: List of BIO tags aligned with tokens
    """
    # Tokenize and get character-to-token mapping
    encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    tokens = encoding.tokens()
    offsets = encoding['offset_mapping']  # [(start_char, end_char) for each token]

    # Initialize all tags as 'O' (Outside)
    tags = ['O'] * len(tokens)

    # Process each annotation
    for annotation in annotations:
        if not annotation.get('label') or not annotation.get('points'):
            continue

        label = annotation['label'][0]  # e.g., 'Skills', 'Education'
        char_start = annotation['points'][0]['start']
        char_end = annotation['points'][0]['end'] + 1  # DataTurks uses inclusive end

        # Find tokens that overlap with this entity
        is_first_token = True
        for idx, (token_start, token_end) in enumerate(offsets):
            # Check if token overlaps with entity span
            if token_start < char_end and token_end > char_start:
                # Assign B- tag for first token, I- for subsequent
                if is_first_token:
                    tags[idx] = f'B-{label}'
                    is_first_token = False
                else:
                    tags[idx] = f'I-{label}'

    return tokens, tags

In [6]:
def preprocess_dataset(data: List[Dict], tokenizer) -> pd.DataFrame:
    """
    Convert raw DataTurks format to a structured DataFrame.

    Args:
        data: List of DataTurks JSON objects
        tokenizer: BERT tokenizer

    Returns:
        DataFrame with columns: 'tokens', 'tags'
    """
    processed_data = []

    for item in data:
        text = item.get('content', '')
        annotations = item.get('annotation', [])

        if not text:
            continue

        tokens, tags = char_to_token_alignment(text, annotations, tokenizer)

        if tokens:  # Only add if we have tokens
            processed_data.append({
                'tokens': tokens,
                'tags': tags
            })

    df = pd.DataFrame(processed_data)
    print(f"Processed {len(df)} documents with tokens and BIO tags")
    return df

## 3. Create Label Mappings and Dataset

In [7]:
def create_label_mappings(df: pd.DataFrame) -> Tuple[Dict, Dict]:
    """
    Create bidirectional mappings between labels and IDs.

    Args:
        df: DataFrame with 'tags' column

    Returns:
        label2id: Dict mapping label strings to integers
        id2label: Dict mapping integers to label strings
    """
    unique_labels = set()
    for tags in df['tags']:
        unique_labels.update(tags)

    unique_labels = sorted(list(unique_labels))
    label2id = {label: idx for idx, label in enumerate(unique_labels)}
    id2label = {idx: label for label, idx in label2id.items()}

    print(f"Found {len(unique_labels)} unique labels:")
    print(unique_labels)

    return label2id, id2label

In [8]:
def tokenize_and_align_labels(examples, tokenizer, label2id, max_length=512):
    """
    Tokenize text and align labels for BERT input.
    Handles special tokens and subword alignment.

    Args:
        examples: Batch from Dataset
        tokenizer: BERT tokenizer
        label2id: Label to ID mapping
        max_length: Maximum sequence length

    Returns:
        Tokenized inputs with aligned labels
    """
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=max_length,
        padding='max_length'
    )

    labels = []
    for i, label_list in enumerate(examples['tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            # Special tokens get label -100 (ignored in loss)
            if word_idx is None:
                label_ids.append(-100)
            # First token of word gets the label
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_list[word_idx]])
            # Subsequent subword tokens get -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

## 4. Evaluation Metrics with Seqeval

Seqeval provides entity-level metrics (not just token accuracy). It correctly handles multi-token entities.

In [9]:
def compute_metrics(eval_pred, id2label):
    """
    Compute entity-level Precision, Recall, and F1 using seqeval.

    Args:
        eval_pred: Tuple of (predictions, labels) from Trainer
        id2label: ID to label mapping

    Returns:
        Dictionary of metrics
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):
        true_label = []
        true_prediction = []

        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                true_label.append(id2label[label_id])
                true_prediction.append(id2label[pred_id])

        true_labels.append(true_label)
        true_predictions.append(true_prediction)

    # Calculate entity-level metrics
    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

## 5. Main Training Pipeline

### Load Data and Initialize Model

In [10]:
import os
import glob

# Configuration
MODEL_NAME = 'bert-base-cased'  # Upgrade to full BERT
MAX_LENGTH = 512                # Double the context window to catch skills at bottom of page
BATCH_SIZE = 8                  # Reduce batch size to fit larger model/context on T4 GPU
LEARNING_RATE = 2e-5            # Lower learning rate for the larger model to prevent overfitting
NUM_EPOCHS = 15
OUTPUT_DIR = './ner_model_output'

# Correct filename identified from directory listing
DATA_FILE = os.path.join(dataturks_resume_entities_for_ner_path, 'Entity Recognition in Resumes.json')
print(f"Using data file: {DATA_FILE}")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer: {MODEL_NAME}")

Using data file: /root/.cache/kagglehub/datasets/dataturks/resume-entities-for-ner/versions/1/Entity Recognition in Resumes.json


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Loaded tokenizer: bert-base-cased


In [11]:
# Load and preprocess data
raw_data = load_dataturks_json(DATA_FILE)
df = preprocess_dataset(raw_data, tokenizer)

# Display sample
print("\nSample processed data:")
print(f"Tokens: {df.iloc[0]['tokens'][:10]}")
print(f"Tags: {df.iloc[0]['tags'][:10]}")

Token indices sequence length is longer than the specified maximum sequence length for this model (1014 > 512). Running this sequence through the model will result in indexing errors


Loaded 220 documents
Processed 220 documents with tokens and BIO tags

Sample processed data:
Tokens: ['A', '##b', '##his', '##he', '##k', 'J', '##ha', 'Application', 'Development', 'Associate']
Tags: ['B-Name', 'I-Name', 'I-Name', 'I-Name', 'I-Name', 'I-Name', 'I-Name', 'B-Designation', 'I-Designation', 'I-Designation']


In [12]:
import os

# List contents of the downloaded dataset directory
print(f"Contents of {dataturks_resume_entities_for_ner_path}:")
for root, dirs, files in os.walk(dataturks_resume_entities_for_ner_path):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

Contents of /root/.cache/kagglehub/datasets/dataturks/resume-entities-for-ner/versions/1:
/root/.cache/kagglehub/datasets/dataturks/resume-entities-for-ner/versions/1/Entity Recognition in Resumes.json


In [13]:
# Create label mappings
label2id, id2label = create_label_mappings(df)
num_labels = len(label2id)
print(f"\nNumber of labels: {num_labels}")

Found 23 unique labels:
['B-College Name', 'B-Companies worked at', 'B-Degree', 'B-Designation', 'B-Email Address', 'B-Graduation Year', 'B-Location', 'B-Name', 'B-Skills', 'B-UNKNOWN', 'B-Years of Experience', 'I-College Name', 'I-Companies worked at', 'I-Degree', 'I-Designation', 'I-Email Address', 'I-Graduation Year', 'I-Location', 'I-Name', 'I-Skills', 'I-UNKNOWN', 'I-Years of Experience', 'O']

Number of labels: 23


In [14]:
# Split dataset
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"\nTrain size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))


Train size: 176
Test size: 44


In [15]:
# Tokenize datasets
train_tokenized = train_dataset.map(
    lambda x: tokenize_and_align_labels(x, tokenizer, label2id, MAX_LENGTH),
    batched=True,
    remove_columns=train_dataset.column_names
)

test_tokenized = test_dataset.map(
    lambda x: tokenize_and_align_labels(x, tokenizer, label2id, MAX_LENGTH),
    batched=True,
    remove_columns=test_dataset.column_names
)

print("Tokenization complete!")

Map:   0%|          | 0/176 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Tokenization complete!


### Initialize BERT Model for Token Classification

In [16]:
# Load pre-trained BERT model
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print(f"Model loaded with {num_labels} labels")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded with 23 labels
Model parameters: 107.74M


### Configure Training Arguments

In [18]:
from transformers import EarlyStoppingCallback, DataCollatorForTokenClassification

# 1. Update Arguments: More epochs, but safer settings
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,           # Lower rate for stability
    per_device_train_batch_size=8, # Fits on T4 GPU with 512 length
    per_device_eval_batch_size=8,
    num_train_epochs=15,          # Set high, let EarlyStopping cut it short
    weight_decay=0.01,            # Regularization to prevent overfitting
    warmup_ratio=0.1,             # 10% warmup prevents early spikes
    fp16=True,                    # CRITICAL: 2x faster training on Colab T4
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,  # Always keeps the "winner" epoch
    metric_for_best_model='f1',
    push_to_hub=False,
    report_to='none'
)

# Define data_collator for dynamic padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 2. Add Callback to Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda eval_pred: compute_metrics(eval_pred, id2label),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stops if no improvement for 3 epochs
)

### Train the Model

In [19]:
# Initialize Trainer
# Define data_collator for dynamic padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda eval_pred: compute_metrics(eval_pred, id2label)
)

print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,1.089418,0.000000,0.000000,0.000000
2,No log,0.726021,0.214815,0.077748,0.114173
3,1.520500,0.567425,0.201220,0.088472,0.122905
4,1.520500,0.475574,0.207317,0.182306,0.194009
5,0.567600,0.441561,0.298361,0.243968,0.268437
6,0.567600,0.395807,0.278689,0.273458,0.276049
7,0.411700,0.381920,0.268868,0.305630,0.286073
8,0.411700,0.373164,0.325926,0.353887,0.339332
9,0.411700,0.362871,0.308889,0.372654,0.337789
10,0.321900,0.358982,0.307036,0.386059,0.342043


TrainOutput(global_step=330, training_loss=0.522382302717729, metrics={'train_runtime': 792.3736, 'train_samples_per_second': 3.332, 'train_steps_per_second': 0.416, 'total_flos': 689954407464960.0, 'train_loss': 0.522382302717729, 'epoch': 15.0})

## 6. Evaluation with Classification Report

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
# Evaluate on test set
print("\n" + "="*50)
print("FINAL EVALUATION ON TEST SET")
print("="*50 + "\n")

predictions = trainer.predict(test_tokenized)
preds = np.argmax(predictions.predictions, axis=2)

# Convert to label strings for seqeval
true_labels = []
true_predictions = []

for prediction, label in zip(preds, predictions.label_ids):
    true_label = []
    true_prediction = []

    for pred_id, label_id in zip(prediction, label):
        if label_id != -100:
            true_label.append(id2label[label_id])
            true_prediction.append(id2label[pred_id])

    true_labels.append(true_label)
    true_predictions.append(true_prediction)

# Print detailed classification report
print(classification_report(true_labels, true_predictions))

# Print overall metrics
print(f"\nOverall Entity-level Metrics:")
print(f"Precision: {precision_score(true_labels, true_predictions):.4f}")
print(f"Recall: {recall_score(true_labels, true_predictions):.4f}")
print(f"F1-Score: {f1_score(true_labels, true_predictions):.4f}")


FINAL EVALUATION ON TEST SET



                     precision    recall  f1-score   support

       College Name       0.00      0.00      0.00        16
Companies worked at       0.26      0.46      0.33        69
             Degree       0.11      0.12      0.11        17
        Designation       0.28      0.35      0.31        85
      Email Address       0.62      0.76      0.68        42
    Graduation Year       0.00      0.00      0.00        14
           Location       0.52      0.53      0.52        57
               Name       0.69      0.67      0.68        46
             Skills       0.02      0.05      0.03        22
Years of Experience       0.00      0.00      0.00         5

          micro avg       0.33      0.42      0.37       373
          macro avg       0.25      0.29      0.27       373
       weighted avg       0.35      0.42      0.38       373


Overall Entity-level Metrics:
Precision: 0.3271
Recall: 0.4236
F1-Score: 0.3692


In [23]:
# Run this after your trainer.evaluate()
import numpy as np

# Get raw predictions
predictions, labels, _ = trainer.predict(test_tokenized)
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens like -100)
true_predictions = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

# Print the first result to compare
print("Model predicted:", true_predictions[0])
print("Actual labels:  ", true_labels[0])

Model predicted: ['B-Name', 'I-Name', 'I-Name', 'I-Name', 'B-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'O', 'B-Companies worked at', 'I-Companies worked at', 'B-Location', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'I-Email Address', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'I-Designation', 'B-Companies worked at', 'O', 'O', 'O', 'O', 'O', 

## 7. Save the Model

In [26]:
import os

SAVE_PATH = "/content/drive/MyDrive/resume_skill_extractor_v1"

# Create directory if it doesn't exist
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# Save using the trainer (saves model + config)
trainer.save_model(SAVE_PATH)

# Save the tokenizer separately
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Model and Tokenizer saved successfully to: {SAVE_PATH}")

✅ Model and Tokenizer saved successfully to: /content/drive/MyDrive/resume_skill_extractor_v1


### Load Models

In [27]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch

# Define path where you saved it
LOAD_PATH = "/content/drive/MyDrive/resume_skill_extractor_v1"

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(LOAD_PATH)

# 2. Load Model
model = AutoModelForTokenClassification.from_pretrained(LOAD_PATH)

# 3. Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("✅ Model and Tokenizer loaded from Drive and moved to", device)

✅ Model and Tokenizer loaded from Drive and moved to cuda


## 8. Inference Function

Create a simple function to extract entities from raw resume text.

In [28]:
def predict_skills(text: str, model, tokenizer, id2label, device='cpu'):
    """
    Extract entities from raw text using the trained NER model.

    Args:
        text: Raw resume text
        model: Trained NER model
        tokenizer: BERT tokenizer
        id2label: ID to label mapping
        device: 'cpu' or 'cuda'

    Returns:
        Dictionary with entity types as keys and lists of extracted entities as values
    """
    model.eval()
    model.to(device)

    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)

    # Convert predictions to labels
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    pred_labels = [id2label[p.item()] for p in predictions[0]]

    # Extract entities
    entities = {}
    current_entity = []
    current_label = None

    for token, label in zip(tokens, pred_labels):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue

        if label.startswith('B-'):
            # Save previous entity
            if current_entity and current_label:
                entity_text = tokenizer.convert_tokens_to_string(current_entity)
                if current_label not in entities:
                    entities[current_label] = []
                entities[current_label].append(entity_text)

            # Start new entity
            current_label = label[2:]
            current_entity = [token]

        elif label.startswith('I-') and current_label == label[2:]:
            current_entity.append(token)

        else:
            # Save and reset
            if current_entity and current_label:
                entity_text = tokenizer.convert_tokens_to_string(current_entity)
                if current_label not in entities:
                    entities[current_label] = []
                entities[current_label].append(entity_text)
            current_entity = []
            current_label = None

    # Don't forget the last entity
    if current_entity and current_label:
        entity_text = tokenizer.convert_tokens_to_string(current_entity)
        if current_label not in entities:
            entities[current_label] = []
        entities[current_label].append(entity_text)

    return entities

## 9. Test Inference on Sample Text

In [29]:
# Example usage
sample_resume = """
John Doe
Software Engineer

Education:
Bachelor of Science in Computer Science, MIT, 2018

Experience:
Senior Developer at Google, 2019-2023
- Developed scalable microservices using Python and Java
- Implemented CI/CD pipelines with Docker and Kubernetes

Skills:
Python, Java, C++, JavaScript, React, Docker, Kubernetes, AWS, Machine Learning, TensorFlow
"""

# Run inference
device = 'cuda' if torch.cuda.is_available() else 'cpu'
extracted_entities = predict_skills(sample_resume, model, tokenizer, id2label, device)

print("\nExtracted Entities:")
print("=" * 50)
for entity_type, entity_list in extracted_entities.items():
    print(f"\n{entity_type}:")
    for entity in entity_list:
        print(f"  - {entity}")


Extracted Entities:

Name:
  - John


## 10. Batch Prediction Function

In [30]:
def batch_predict_resumes(resume_texts: List[str], model, tokenizer, id2label, device='cpu'):
    """
    Extract entities from multiple resumes.

    Args:
        resume_texts: List of resume text strings
        model: Trained NER model
        tokenizer: BERT tokenizer
        id2label: ID to label mapping
        device: 'cpu' or 'cuda'

    Returns:
        List of dictionaries containing extracted entities
    """
    results = []
    for text in resume_texts:
        entities = predict_skills(text, model, tokenizer, id2label, device)
        results.append(entities)
    return results

# Example batch usage
# resume_list = [resume1, resume2, resume3]
# batch_results = batch_predict_resumes(resume_list, model, tokenizer, id2label, device)

In [31]:
# Install Gradio library
!pip install gradio

In [32]:
import gradio as gr

def gradio_predict_skills(text: str) -> str:
    """
    Wrapper function for predict_skills to be used with Gradio.
    Formats the output for display in the Gradio interface.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    extracted_entities = predict_skills(text, model, tokenizer, id2label, device)

    if not extracted_entities:
        return "No entities extracted."

    output_str = ""
    for entity_type, entity_list in extracted_entities.items():
        output_str += f"**{entity_type}:**\n"
        for entity in entity_list:
            output_str += f"- {entity}\n"
        output_str += "\n"
    return output_str


In [34]:
import re

# 1. Define a list of known skills (The "Safety Net")
KNOWN_SKILLS = ["Python", "SQL", "Machine Learning", "Java", "C++", "Data Analysis", "NLP"]

def hybrid_prediction(text):
    results = []

    # --- STEP 1: BERT Model Prediction (The "Brain") ---
    # Ensure inputs are moved to the same device as the model
    device = next(model.parameters()).device # Get the current device of the model
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        predictions = torch.argmax(logits, dim=2)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [model.config.id2label[t.item()] for t in predictions[0]]

    # Extract BERT results
    for token, label in zip(tokens, labels):
        if label != 'O' and token not in ['[CLS]', '[SEP]', '[PAD]']:
            # Clean up token (remove ##)
            clean_word = token.replace("##", "")
            results.append((clean_word, label))

    # --- STEP 2: Keyword Matching (The "Backup") ---
    # If BERT missed the skills, we catch them here
    for skill in KNOWN_SKILLS:
        # Simple check: is the skill in the text?
        if re.search(r'\b' + re.escape(skill) + r'\b', text, re.IGNORECASE):
            # Check if we already found it to avoid duplicates
            if not any(r[0].lower() == skill.lower() for r in results):
                results.append((skill, "B-Skills (Found by Keyword)"))

    return results

# --- TEST IT ---
test_text = """
Name: Jane Smith
Email: jane@example.com
Skills: I am good at Python, SQL and Machine Learning.
"""

print(f"{'ENTITY':<20} | {'LABEL'}")
print("-" * 40)
final_output = hybrid_prediction(test_text)
for entity, label in final_output:
    print(f"{entity:<20} | {label}")

ENTITY               | LABEL
----------------------------------------
Name                 | B-Name
Python               | B-Skills (Found by Keyword)
SQL                  | B-Skills (Found by Keyword)
Machine Learning     | B-Skills (Found by Keyword)


In [35]:
import re

KNOWN_SKILLS = ["Python", "SQL", "Machine Learning", "Java", "C++", "Data Analysis", "NLP", "Pandas", "TensorFlow"]

def hybrid_prediction(text):
    results = []

    # --- STEP 1: BERT Model Prediction ---
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
        predictions = torch.argmax(logits, dim=2)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [model.config.id2label[t.item()] for t in predictions[0]]

    for token, label in zip(tokens, labels):
        if label != 'O' and token not in ['[CLS]', '[SEP]']:
            clean_word = token.replace("##", "")

            # CLEAN UP THE LABEL: Remove "B-" and "I-" prefixes
            # This turns "B-Designation" into "Designation"
            clean_label = label.replace("B-", "").replace("I-", "")

            results.append((clean_word, clean_label))

    # --- STEP 2: Keyword Matching ---
    for skill in KNOWN_SKILLS:
        if re.search(r'\b' + re.escape(skill) + r'\b', text, re.IGNORECASE):
            # Check for duplicates
            if not any(r[0] == skill for r in results):
                # CHANGE IS HERE: We simply call it "Skill" now
                results.append((skill, "Skill"))

    return results

In [36]:
import gradio as gr
import torch
import re

# 1. Define the Safety Net (Keywords to catch if BERT misses them)
KNOWN_SKILLS = [
    "Python", "SQL", "Java", "C++", "Machine Learning", "Data Analysis",
    "NLP", "Pandas", "TensorFlow", "AWS", "Docker", "Kubernetes",
    "React", "JavaScript", "HTML", "CSS", "Project Management"
]

def hybrid_prediction(text):
    """
    Combines BERT-base model predictions with keyword matching.
    """
    results = []

    # --- STEP 1: BERT Model Prediction ---
    # We use the global 'model' and 'tokenizer' from your training steps
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()

    # Tokenize (Max Length 512 for full context)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        predictions = torch.argmax(logits, dim=2)

    # Convert IDs to Tokens and Labels
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [model.config.id2label[t.item()] for t in predictions[0]]

    # Extract BERT entities
    current_entity = []
    current_label = None

    for token, label in zip(tokens, labels):
        # Skip special tokens
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue

        # Handle Subwords (Remove '##' prefix)
        clean_word = token.replace("##", "")

        if label.startswith("B-"):
            # Save previous entity if exists
            if current_entity:
                results.append(("".join(current_entity), current_label))
            # Start new entity
            current_entity = [clean_word]
            current_label = label.replace("B-", "") # Clean label name

        elif label.startswith("I-") and current_label == label.replace("I-", ""):
            # Continue current entity
            current_entity.append(clean_word)

        else:
            # Save and reset
            if current_entity:
                results.append(("".join(current_entity), current_label))
            current_entity = []
            current_label = None

    # --- STEP 2: Keyword Matching Safety Net ---
    # Check for known skills that the model might have missed
    text_lower = text.lower()
    for skill in KNOWN_SKILLS:
        # Use regex to find whole words only
        if re.search(r'\b' + re.escape(skill.lower()) + r'\b', text_lower):
            # Only add if we haven't found a similar entity already
            # (Simple check to avoid duplicates)
            if not any(skill.lower() in r[0].lower() for r in results):
                results.append((skill, "Skills (Keyword Match)"))

    return results

def format_output_for_gradio(text):
    if not text.strip():
        return "Please enter resume text."

    entities = hybrid_prediction(text)

    if not entities:
        return "No specific entities detected. Try pasting a more detailed resume."

    # Group by category for cleaner display
    grouped = {}
    for word, label in entities:
        if label not in grouped:
            grouped[label] = set() # Use set to remove exact duplicates
        grouped[label].add(word)

    # Format output string
    output_str = "### 📄 Extraction Results\n"
    output_str += f"**Model used:** {MODEL_NAME} (Context: 512 tokens)\n\n"

    for label, items in grouped.items():
        output_str += f"**{label}:**\n"
        for item in sorted(items):
            output_str += f"• {item}\n"
        output_str += "\n"

    return output_str

# 3. Define Examples for the UI
examples = [
    ["John Doe\nSoftware Engineer\n\nExperience:\nSenior Python Developer at Tech Corp (2019-2023)\n- Built microservices using Flask and Docker.\n- Managed AWS infrastructure.\n\nSkills: Python, SQL, Kubernetes, React, Agile."],
    ["Jane Smith\nData Scientist\n\nEducation:\nM.S. in Computer Science, MIT\n\nProjects:\n- NLP Analysis using BERT and Transformers.\n- Visualized data with Tableau and D3.js."]
]

# 4. Launch Interface
iface = gr.Interface(
    fn=format_output_for_gradio,
    inputs=gr.Textbox(lines=15, placeholder="Paste Resume Text Here...", label="Resume Content"),
    outputs=gr.Markdown(label="Detected Entities"),
    title="🚀 AI Resume Skill Extractor (BERT-Base)",
    description="Extracts Skills, Experience, and Education using a fine-tuned BERT model + Keyword Safety Net.",
    examples=examples,
    theme="soft"
)

iface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8b5c3728128c2ba405.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8b5c3728128c2ba405.gradio.live


## Summary

This notebook provides a complete pipeline for training a BERT-based NER model for resume skill extraction:

1. ✅ **Data Loading**: Handles DataTurks JSON format
2. ✅ **Preprocessing**: Robust character-to-token alignment with BIO tagging
3. ✅ **Model**: BERT-base-cased for case-sensitive entity recognition
4. ✅ **Training**: Hugging Face Trainer API with proper hyperparameters
5. ✅ **Evaluation**: Entity-level metrics using seqeval
6. ✅ **Inference**: Simple prediction function for new text

### Next Steps:
- Fine-tune hyperparameters (learning rate, batch size, epochs)
- Try other models (RoBERTa, DistilBERT, domain-specific BERT)
- Implement data augmentation for rare entity types
- Add confidence scores to predictions
- Deploy as REST API using FastAPI or Flask